In [0]:
%pip install sentence-transformers faiss-cpu
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import faiss
import os
from sentence_transformers import SentenceTransformer

In [0]:
# 1. Pull the Delta Table into driver memory as the data is very small
print("Loading 'silver_schemes' from Unity Catalog...")
df_schemes = spark.table("nyaya_hackathon.schemes_app.silver_schemes").toPandas()

Loading 'silver_schemes' from Unity Catalog...


In [0]:
# 2. Load the CPU-optimized embedding model
# all-MiniLM-L6-v2 is incredibly fast, takes ~90MB of RAM, and outputs 384D vectors
print("Loading MiniLM embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

Loading MiniLM embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [0]:
# 3. Generate Embeddings
print(f"Vectorizing {len(df_schemes)} scheme contexts...")
# Convert to numpy array as required by FAISS
contexts = df_schemes['search_context'].tolist()
embeddings = embedder.encode(contexts, convert_to_numpy=True, show_progress_bar=True)

Vectorizing 3400 scheme contexts...


Batches:   0%|          | 0/107 [00:00<?, ?it/s]

In [0]:
# 4. Build the FAISS Index
# We use IndexFlatL2 for exact L2 distance search (perfect for small datasets)
dimension = embeddings.shape[1]  # Should be 384
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print(f"FAISS index built with {index.ntotal} vectors.")

FAISS index built with 3400 vectors.


In [0]:
# 5. Create storage directory in Volumes (serverless-compatible)
volume_path = "/Volumes/nyaya_hackathon/schemes_app/app_storage"
os.makedirs(volume_path, exist_ok=True)

# 6. Save the Index and Mapping to Volumes
# The App (Phase 3) will load these two files into RAM when a user connects
faiss_path = os.path.join(volume_path, "scheme_index.bin")
mapping_path = os.path.join(volume_path, "scheme_mapping.csv")

faiss.write_index(index, faiss_path)

# Incorrect - Save the DataFrame to easily retrieve the exact scheme details (benefits, application) later
# Save as a clean, universal CSV (dropping the index)
df_schemes.to_csv(mapping_path, index=False)
print(f" Clean CSV saved to: {mapping_path}")

print(f"Success: Assets saved to Volumes.")
print(f"Index: {faiss_path}")
print(f"Mapping: {mapping_path}")

 Clean CSV saved to: /Volumes/nyaya_hackathon/schemes_app/app_storage/scheme_mapping.csv
Success: Assets saved to Volumes.
Index: /Volumes/nyaya_hackathon/schemes_app/app_storage/scheme_index.bin
Mapping: /Volumes/nyaya_hackathon/schemes_app/app_storage/scheme_mapping.csv
